# 02 · 因子构建与预处理

> 本 notebook 是《量化研究入门学习资料》第 X 章的可运行配套。
> 数据源：`data/csv/`。运行前请先执行 `python scripts/generate_data.py` 生成数据。

## 目标
从清洗后的价格矩阵重算价格类因子（MOM60/REV5/VOL20），从 `stocks_basic.csv` 计算规模与估值因子，并对全部 8 个因子跑「去极值 → 行业中性化 → z-score」管道，与 `data/csv/factors.csv` 的 z 列对照。

> 注：ROE/GROW/TURN 属于基本面/另类数据，教学上直接采用数据文件中的原始值（获取这类数据本身不是本章重点）。

In [ ]:
import pandas as pd, numpy as np

prices = pd.read_csv("data/csv/prices.csv", parse_dates=["date"])
basic = pd.read_csv("data/csv/stocks_basic.csv", parse_dates=["delist_date"])
factors = pd.read_csv("data/csv/factors.csv", parse_dates=["date"])

# 清洗 → 宽表（date × code）
df = prices.drop_duplicates(["code", "date"]).sort_values(["code", "date"])
df["close"] = df.groupby("code")["close"].ffill()
for _, r in basic.dropna(subset=["delist_date"]).iterrows():
    df = df.drop(df[(df["code"] == r["code"]) & (df["date"] > r["delist_date"])].index)
close = df.pivot(index="date", columns="code", values="close")
close.index = pd.to_datetime(close.index)

INDUSTRIES = ["银行", "非银金融", "医药生物", "电子", "食品饮料",
              "机械设备", "汽车", "电力公用"]
rebal = factors.groupby("date")["code"].count().index      # 60 个调仓日

In [ ]:
# ---- 因子原始值重算 ----
codes = list(close.columns)
shares = basic.set_index("code").loc[codes, "shares"].values
bvps = basic.set_index("code").loc[codes, "bvps"].values
ind_map = dict(zip(basic["code"], basic["industry"]))

rows = []
for d0 in rebal:
    i0 = close.index.get_loc(d0)
    px = close.iloc[i0].values
    px_60 = close.iloc[max(0, i0 - 60)].values
    px_5 = close.iloc[max(0, i0 - 5)].values
    vol20 = -close.iloc[max(0, i0 - 20):i0].values.std(axis=0)   # ddof=0，与生成脚本一致
    mom60 = px / px_60 - 1
    rev5 = -(px / px_5 - 1)
    size = np.log(np.where(px > 0, px * shares, np.nan))
    rows.append({"date": d0, "code": codes,
                 "MOM60": mom60, "REV5": rev5, "VOL20": vol20, "SIZE": size})
long = pd.concat([pd.DataFrame({**{"date": r["date"], "code": r["code"]},
                                **{k: v for k, v in r.items() if k not in ("date", "code")}})
                  for r in rows], ignore_index=True)
# 对照 factors.csv 的价格类因子原始值
for col in ["MOM60", "REV5", "VOL20", "SIZE"]:
    ref = factors[["date", "code", col]].merge(long, on=["date", "code"], suffixes=("", "_r"))
    ok = np.allclose(ref[col].fillna(0), ref[col + "_r"].fillna(0), atol=1e-6)
    print(f"{col:6s} 重算对照: {'PASS' if ok else 'FAIL'}")

In [ ]:
# ---- 预处理管道：winsorize → 行业中性化 → z-score ----
def factor_pipeline(z_mat, industries):
    """与生成脚本同款管道（教学版，逐期截面处理）"""
    T, S = z_mat.shape
    out = np.full_like(z_mat, np.nan)
    dum = pd.get_dummies(industries).values.astype(float)
    X = np.column_stack([np.ones(S), dum[:, :-1]])
    for t in range(T):
        x = z_mat[t]; m = np.isfinite(x)
        if m.sum() < 10: continue
        xm, sd = x[m].mean(), x[m].std()
        if sd == 0 or not np.isfinite(sd): continue
        xw = np.clip(x, xm - 3 * sd, xm + 3 * sd)             # 去极值
        beta, *_ = np.linalg.lstsq(X[m], xw[m], rcond=None)
        resid = xw - X @ beta                                 # 行业中性化
        r = resid[m].std()
        if r == 0 or not np.isfinite(r): continue
        out[t] = (resid - resid[m].mean()) / r                # z-score
    return out

inds = np.array([ind_map[c] for c in codes])
z_ok = {}
for col in ["EP", "SIZE", "MOM60", "REV5", "VOL20", "TURN", "ROE", "GROW"]:
    # 原始值：价格类用重算结果，基本面类用 factors.csv
    if col in ("MOM60", "REV5", "VOL20", "SIZE"):
        raw = long.pivot(index="date", columns="code", values=col).reindex(rebal).values
    else:
        raw = factors.pivot(index="date", columns="code", values=col).values
    z_calc = factor_pipeline(raw, inds)
    z_ref = factors.pivot(index="date", columns="code", values="z_" + col).values
    same = (np.isnan(z_calc) == np.isnan(z_ref))
    ok = same.all() and np.allclose(np.nan_to_num(z_calc), np.nan_to_num(z_ref), atol=1e-4)
    z_ok[col] = ok
    print(f"z_{col:6s} 管道重算对照: {'PASS' if ok else 'FAIL'}")
assert all(z_ok.values()), "存在不一致的因子" 